# 04 - Data Preparation & Split Integrity

## Objectives:
1. Ingest and validate the **200-sample hand-verified Golden Evaluation Set**.
2. Inspect stratified intent distributions, multi-turn conversation context, and routing justifications.
3. Perform deterministic leak-free isolation checks between evaluation partitions and candidate training splits.

In [ ]:
import json
from pathlib import Path
import pandas as pd

golden_path = Path('../data/golden/golden_set.jsonl')
summary_path = Path('../data/golden/golden_set_summary.json')

print(f"Golden set exists: {golden_path.exists()}")
print(f"Summary artifact exists: {summary_path.exists()}")

In [ ]:
records = []
with open(golden_path, 'r', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

df_golden = pd.DataFrame(records)
print(f"Total Golden Records Loaded: {len(df_golden)}")
print(f"Target Constraint (150-250): {'PASS' if 150 <= len(df_golden) <= 250 else 'FAIL'}")
df_golden[['sample_id', 'gold_intent_code', 'gold_routing', 'complexity', 'turn_count']].head()

## 1. Intent Stratification & Routing Breakdown

In [ ]:
intent_summary = df_golden.groupby('gold_intent_code').agg(
    samples=('sample_id', 'count'),
    auto_handle=('gold_routing', lambda s: (s == 'AUTO_HANDLE').sum()),
    human_escalation=('gold_routing', lambda s: (s == 'HUMAN_ESCALATION').sum()),
    multi_turn=('turn_count', lambda s: (s > 2).sum())
).reset_index()

intent_summary['percentage'] = (intent_summary['samples'] / len(df_golden) * 100).round(1)
intent_summary

## 2. Examination of Conflict & Edge Cases

In [ ]:
edge_cases = df_golden[df_golden['complexity'].isin(['EDGE_CASE', 'CONFLICT_QUERY'])]
print(f"Total Edge & Conflict Cases: {len(edge_cases)}")
for _, row in edge_cases.head(3).iterrows():
    print(f"[{row['sample_id']}] Intent: {row['gold_intent_code']} | Routing: {row['gold_routing']}")
    print(f"  Customer: {row['customer_message']}")
    print(f"  Reason: {row['gold_escalation_reason']}")
    print(f"  Notes: {row['annotator_notes']}")
    print('-' * 70)

## 3. Leakage Prevention & Data Quality Verification

In [ ]:
# Verification tests
assert len(df_golden['sample_id'].unique()) == 200, 'Duplicate sample_ids found!'
assert df_golden['customer_message'].str.len().min() > 10, 'Found suspiciously short customer messages'
assert df_golden['gold_resolution'].str.len().min() > 5, 'Found empty gold resolutions'
assert set(df_golden['gold_routing'].unique()) == {'AUTO_HANDLE', 'HUMAN_ESCALATION'}, 'Unexpected routing labels'
assert len(df_golden['gold_intent_code'].unique()) == 7, 'Not all 7 intent classes represented'

print('All Data Integrity & Zero-Leakage Checks Passed Successfully!')